In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
application_test = pd.read_csv("home-credit-default-risk/application_test.csv")
application_train = pd.read_csv("home-credit-default-risk/application_train.csv")
bureau = pd.read_csv("home-credit-default-risk/bureau.csv")
bureau_balance = pd.read_csv("home-credit-default-risk/bureau_balance.csv")
credit_card_balance = pd.read_csv("home-credit-default-risk/credit_card_balance.csv")
installments_payments = pd.read_csv("home-credit-default-risk/installments_payments.csv")
POS_CASH_balance = pd.read_csv("home-credit-default-risk/POS_CASH_balance.csv")
previous_application = pd.read_csv("home-credit-default-risk/previous_application.csv")
sample_submission = pd.read_csv("home-credit-default-risk/sample_submission.csv")

In [21]:
application_train.shape

(307511, 122)

In [22]:
#bureau.csv

In [23]:
bureau.shape

(1716428, 17)

In [24]:
bureau.columns

Index(['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY',
       'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE',
       'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG',
       'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT',
       'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE',
       'AMT_ANNUITY'],
      dtype='object')

In [25]:
bureau["SK_ID_CURR"].nunique()

305811

In [27]:
bureau.head(15)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.00,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.00,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0,180000.00,71017.38,108982.62,0.0,Credit card,-31,NaN
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0,42103.80,42103.80,0.00,0.0,Consumer credit,-22,NaN
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0,76878.45,0.00,0.00,0.0,Consumer credit,-1710,NaN
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0,103007.70,0.00,0.00,0.0,Consumer credit,-840,NaN
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0,4500.00,0.00,0.00,0.0,Credit card,-690,NaN


In [28]:
bureau["CREDIT_ACTIVE"].value_counts()

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

In [32]:
bureau.isnull().sum().sort_values(ascending=False)

DAYS_ENDDATE_FACT         633645
AMT_CREDIT_SUM_LIMIT      591767
AMT_CREDIT_SUM_DEBT       257669
DAYS_CREDIT_ENDDATE       105544
SK_ID_CURR                     0
SK_ID_BUREAU                   0
CREDIT_ACTIVE                  0
CREDIT_CURRENCY                0
DAYS_CREDIT                    0
CREDIT_DAY_OVERDUE             0
CNT_CREDIT_PROLONG             0
AMT_CREDIT_SUM                 0
AMT_CREDIT_SUM_OVERDUE         0
CREDIT_TYPE                    0
DAYS_CREDIT_UPDATE             0
dtype: int64

In [30]:
#AMT_ANNUITY und AMT_CREDIT_MAX_OVERDUE haben mehr als 50% NA Werte
bureau.drop(columns=['AMT_ANNUITY', "AMT_CREDIT_MAX_OVERDUE"], inplace=True)

In [31]:
#AMT_CREDIT_SUM hat nur 13 NA Werte. Diese Zeilen droppen wir einfach
bureau = bureau.dropna(subset=["AMT_CREDIT_SUM"])

In [33]:
bureau["CREDIT_ACTIVE"].value_counts()

CREDIT_ACTIVE
Closed      1079268
Active       630599
Sold           6527
Bad debt         21
Name: count, dtype: int64

In [34]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    bureau_loan_count=("SK_ID_BUREAU", "count"),    #Anzahl Kredite
    bureau_active_count=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()), #Anzahl Aktive Kredite
    bureau_total_credit=("AMT_CREDIT_SUM", "sum"),  #Gesamtkreditehöhe
    bureau_total_debt=("AMT_CREDIT_SUM_DEBT", "sum"),   #Gesamtschulden
    bureau_current_debt=("AMT_CREDIT_SUM_OVERDUE", "sum"),  #Aktuelle Schulden

    bureau_avg_credit_age=("DAYS_CREDIT", "mean"),       # wie lange zurück die Kredite sind
    bureau_most_recent_credit=("DAYS_CREDIT", "max"),    # jüngster Kredit
    bureau_oldest_credit=("DAYS_CREDIT", "min"),         # ältester Kredit

    bureau_avg_credit=("AMT_CREDIT_SUM", "mean"),
    bureau_max_credit=("AMT_CREDIT_SUM", "max"),
).reset_index()
print(application_train.shape)
application_train = application_train.merge(bureau_agg, on="SK_ID_CURR", how="left")

(307511, 122)


In [35]:
print(bureau_agg.shape)

(305809, 11)


In [41]:
application_train.describe()

,SK_ID_CURR,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,...,bureau_loan_count,bureau_active_count,bureau_total_credit,bureau_total_debt,bureau_current_debt,bureau_avg_credit_age,bureau_most_recent_credit,bureau_oldest_credit,bureau_avg_credit,bureau_max_credit
count,307511.000000,307511.000000,307511.000000,3.075110e+05,3.075110e+05,307499.000000,3.072330e+05,307511.000000,307511.000000,252137.000000,...,307511.000000,307511.000000,3.075110e+05,3.075110e+05,3.075110e+05,263490.000000,263490.000000,263490.000000,3.075110e+05,3.075110e+05
mean,278180.518577,0.080729,0.417052,1.687979e+05,5.990260e+05,27108.573909,5.383962e+05,0.020868,-16036.995067,-2384.169325,...,4.765104,1.762269,1.675834e+06,5.489321e+05,1.911071e+02,-1083.052626,-489.303757,-1762.381570,3.239570e+05,8.362866e+05
std,102790.175348,0.272419,0.722121,2.371231e+05,4.024908e+05,14493.737315,3.694465e+05,0.013831,4363.988632,2338.360162,...,4.496200,1.804890,3.858106e+06,1.529044e+06,1.527103e+04,563.324590,537.572988,863.856997,8.359404e+05,1.962377e+06
min,100002.000000,0.000000,0.000000,2.565000e+04,4.500000e+04,1615.500000,4.050000e+04,0.000290,-25229.000000,-17912.000000,...,0.000000,0.000000,0.000000e+00,-6.981558e+06,0.000000e+00,-2922.000000,-2922.000000,-2922.000000,0.000000e+00,0.000000e+00
25%,189145.500000,0.000000,0.000000,1.125000e+05,2.700000e+05,16524.000000,2.385000e+05,0.010006,-19682.000000,-3175.000000,...,1.000000,0.000000,1.539495e+05,0.000000e+00,0.000000e+00,-1434.000000,-620.000000,-2583.000000,6.705532e+04,9.933167e+04
50%,278202.000000,0.000000,0.000000,1.471500e+05,5.135310e+05,24903.000000,4.500000e+05,0.018850,-15750.000000,-1648.000000,...,4.000000,1.000000,7.110000e+05,8.757900e+04,0.000000e+00,-1050.571429,-300.000000,-1827.000000,1.587308e+05,3.240000e+05
75%,367142.500000,0.000000,1.000000,2.025000e+05,8.086500e+05,34596.000000,6.795000e+05,0.028663,-12413.000000,-767.000000,...,7.000000,3.000000,1.970993e+06,5.348462e+05,0.000000e+00,-663.777778,-143.000000,-1035.000000,3.444591e+05,9.000000e+05
max,456255.000000,1.000000,19.000000,1.170000e+08,4.050000e+06,258025.500000,4.050000e+06,0.072508,-7489.000000,0.000000,...,116.000000,32.000000,1.017958e+09,3.344983e+08,3.756681e+06,-2.000000,0.000000,-2.000000,1.980723e+08,3.960000e+08


In [37]:
print(application_train.shape)
print(application_train[["bureau_loan_count"]].count())

(307511, 132)
bureau_loan_count    263490
dtype: int64


In [39]:
#bei diesen Spalten können wir die NA Werte getrost durch 0 ersetzen
bureau_cols = ["bureau_loan_count", "bureau_active_count", "bureau_total_credit",
               "bureau_total_debt", "bureau_current_debt", "bureau_avg_credit",
               "bureau_max_credit"]

application_train[bureau_cols] = application_train[bureau_cols].fillna(0)

In [40]:
#es gab eine Fehlerhafte Zeile wo jemand seit 1000 Jahren angestellt war
application_train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

In [ ]:
#bureau_balance

In [42]:
bureau_balance.shape

(27299925, 3)

In [43]:
bureau_balance.columns

Index(['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS'], dtype='object')

In [44]:
bureau_balance.describe()

,SK_ID_BUREAU,MONTHS_BALANCE
count,2.729992e+07,2.729992e+07
mean,6.036297e+06,-3.074169e+01
std,4.923489e+05,2.386451e+01
min,5.001709e+06,-9.600000e+01
25%,5.730933e+06,-4.600000e+01
50%,6.070821e+06,-2.500000e+01
75%,6.431951e+06,-1.100000e+01
max,6.842888e+06,0.000000e+00


In [45]:
bureau_balance["STATUS"].value_counts()

STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

In [46]:
bureau_balance.isnull().sum().sort_values(ascending=False)

SK_ID_BUREAU      0
MONTHS_BALANCE    0
STATUS            0
dtype: int64